In [1]:
"""
PIPELINE STAGE: Multimodal Integration & Longitudinal Data Consolidation
TIMELINE: 2020 - 2024 (Full 5-Year Temporal Window)
INPUTS: 
    - Master Energy Database: processed_data/steps/final_2020_2024.xlsx 
    - Sequential Population CSV: raw_data/population/population.csv
OUTPUT: processed_data/final/master_energy_demographics_2020_2024.xlsx

1. METHODOLOGICAL SCOPE:
    The pipeline is calibrated for a 60-month longitudinal analysis (Jan 2020 - Dec 2024). 
    This allows for the examination of energy consumption elasticity across significant 
    socioeconomic shifts, including pandemic-induced lockdowns and subsequent recoveries.

2. DATA SOURCE COMPLEXITY:
    - Energy Segments: Disaggregated data (Residential, Industrial, Agricultural, etc.) 
      mined from unstructured Word and Excel sources.
    - Demographic Context: Derived from a non-standard horizontal sequential CSV, 
      pivoted into a relational format using context-aware Python parsing.

3. INTEGRATION STRATEGY:
    - Temporal Logic: Annual population metrics for each of the 5 years are mapped 
      to their respective 12-month clusters via [Plate, Year] composite keys.
    - Structural Integrity: Ensures that for each province (81) and each year (5), 
      the population denominator is consistently applied to all energy sub-sectors.

4. RESEARCH CAPABILITY:
    The resulting master dataset serves as the primary backbone for:
    - Per-Capita Consumption Trends (2020-2024)
    - Comparative Sectoral Impact Analysis (Pre vs. Post Pandemic)
    - Regional Energy Efficiency Benchmarking
"""
import pandas as pd
import os
import re

def nufus_yapilandirma_v2():
    # Dosya Yolları
    IN_PATH = os.path.join("..", "..", "raw_data", "population", "population.csv")
    OUT_PATH = os.path.join("..", "..", "processed_data", "steps", "08_a_cleaned_population.xlsx")

   
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    records = []
    
    try:
        # Dosyayı oku ve içeriği normalize et
        with open(IN_PATH, 'r', encoding='iso-8859-9') as f:
            content = f.read()
            # Satır sonlarını virgüle çevirip temiz bir liste oluştur
            parts = [p.strip() for p in content.replace('\n', ',').replace('\r', ',').split(',') if p.strip()]

        current_year = None
        i = 0
        while i < len(parts):
            item = parts[i]
            
            # Yıl tespiti (Örn: 2020)
            if re.match(r'^20\d{2}$', item):
                current_year = int(item)
                i += 1
                continue
            
            # Veri bloğu tespiti (Örn: Adana-1)
            if current_year and '-' in item:
                if i + 1 < len(parts):
                    geo_info = item
                    pop_count = parts[i+1]
                    
                    try:
                        prov, plate = geo_info.split('-')
                        if pop_count.isdigit():
                            records.append({
                                'Plate': int(plate),      # 1. Sırada
                                'Year': current_year,     # 2. Sırada
                                'Province': prov.strip().upper(), # 3. Sırada
                                'Population': int(pop_count)     # 4. Sırada
                            })
                            i += 2
                            continue
                    except:
                        pass
            i += 1

        if not records:
            print("HATA: Veri işlenemedi.")
            return

        # DataFrame oluştur
        df_pop = pd.DataFrame(records)

        # 1. İSTEK: Sütun yerlerini değiştir (Plate, Year, Province, Population)
        # Liste yapısı ile sütun sırasını zorunlu kılıyoruz
        df_pop = df_pop[['Plate', 'Year', 'Province', 'Population']]
        
        # Mükerrerleri temizle
        df_pop = df_pop.drop_duplicates(subset=['Plate', 'Year'], keep='last')
        
        # 2. İSTEK: Plate ve Year sütunlarına göre artan (A-Z) sırala
        df_pop = df_pop.sort_values(by=['Plate', 'Year'], ascending=[True, True])
        
        # Excel'e aktar
        df_pop.to_excel(OUT_PATH, index=False)
        
        print("\n" + "="*40)
        print(f"İŞLEM BAŞARIYLA TAMAMLANDI")
        print(f"Toplam Satır: {len(df_pop)}")
        print(f"Sıralama: Plate ve Year bazlı artan")
        print(f"Dosya Yolu: {OUT_PATH}")
        print("="*40)

    except Exception as e:
        print(f"BEKLENMEDİK HATA: {e}")

if __name__ == "__main__":
    nufus_yapilandirma_v2()


İŞLEM BAŞARIYLA TAMAMLANDI
Toplam Satır: 405
Sıralama: Plate ve Year bazlı artan
Dosya Yolu: ..\..\processed_data\steps\08_a_cleaned_population.xlsx
